In [4]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.8/33.8 MB 8.8 MB/s eta 0:00:0000:0100:01m

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import mysql.connector as sql
import pandas as pd

# MySQL DB 연결
db_connection = sql.connect(
    host='127.0.0.1', 
    database='orders', 
    user='dev', 
    password='pwd'
)
# orders 테이블에서 sessionId와 customerId를 조회하여 DataFrame으로 로드
query = "SELECT sessionId, customerId FROM orders"
df = pd.read_sql(query, con=db_connection)
df.head()  # 데이터 확인 (상위 5행)


/tmp/ipykernel_2377/1200567215.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=db_connection)


,sessionId,customerId
0,01GZWETRHAHNKXQR2PDRP1ZEJC,01GYNFXQNDD0ATWHDFXCBHD5XG
1,01GZWETRHAHNKXQR2PDRP1ZEJC,01GYNFXQNDD0ATWHDFXCBHD5XG
2,01GZWG6V85M35RWMVVT0R8BW2T,01GYNFXQNDD0ATWHDFXCBHD5XG
3,01GZWKZ87MX12B4PFNZN2RDMPH,01GZWKZZJXDT4H4SAZ068CKYPR
4,01GZWKZ87MX12B4PFNZN2RDMPH,01GYNFXQNDD0ATWHDFXCBHD5XG


In [6]:
from itertools import combinations

# 한 세션 내 모든 고객 페어(쌍) 생성 함수
def combine_customers(customers):
    # customers는 해당 session의 customerId 시리즈; set으로 변환해 고유 고객만 추출
    pairs = list(combinations(set(customers), 2))  # 2명씩 가능한 모든 조합
    return pd.Series(pairs)

# sessionId별로 customerId 그룹화 후, 각 그룹에 대해 고객 쌍 계산
pairs_series = df.groupby('sessionId')['customerId'].apply(combine_customers)
pairs_series.head()  # 각 세션에서 생성된 (고객1, 고객2) 쌍 확인


sessionId                    
01GZWKZ87MX12B4PFNZN2RDMPH  0    (01GYNFXQNDD0ATWHDFXCBHD5XG, 01GZWKZZJXDT4H4SA...
01GZZ0BKVQ3BHJ4GE7MM4FV4HB  0    (01GYNFXQNDD0ATWHDFXCBHD5XG, 01GZZ0C5HVG9049C8...
01GZZ87X94YDEKQFYMTXDXMGKS  0    (01GZZ89GETMF960XK25J96E785, 01GZZ89SHGZJM84N3...
01H01TE0NJNCX8D52M55FERVYF  0    (01H01TEQMXDXGW2GZK2M1HGF26, 01GYNFXQNDD0ATWHD...
01H01VYBZ7EV2VCS4T5BFVC70V  0    (01GYNFXQNDD0ATWHDFXCBHD5XG, 01H01W0CA33HV7EGV...
Name: customerId, dtype: object

In [8]:
# 전체 세션에서 고객 쌍별 동행 횟수 계산
pair_counts = pairs_series.value_counts().reset_index()
pair_counts.columns = ['pair', 'timesTogether']

# 'pair' 컬럼에서 고객1, 고객2로 분해
pair_counts[['customer1', 'customer2']] = pd.DataFrame(pair_counts['pair'].tolist(), index=pair_counts.index)
pair_counts.drop(columns='pair', inplace=True)

pair_counts.head()


,timesTogether,customer1,customer2
0,18,01GYNFXQNDD0ATWHDFXCBHD5XG,01H7HNC3GKV65TAG2H3SGW7A0H
1,16,01H8DZZ71SYRKB4868B0WHV68E,01H7HNC3GKV65TAG2H3SGW7A0H
2,14,01H7HNC3GKV65TAG2H3SGW7A0H,01H847J2DGX5KEKMX386TKYGY1
3,13,01J1V0DRD1RMPF5JFPP2CKCC83,01J1VG1118TSY4KR1R6PJYK3V9
4,11,01H8DZZ71SYRKB4868B0WHV68E,01H7S0P3HK62VATNYDHADQPHKJ


# 가장 많이 등장한 CustomerId

In [12]:
df['customerId'].value_counts()

customerId
01GYNFXQNDD0ATWHDFXCBHD5XG    1142
01H7HNC3GKV65TAG2H3SGW7A0H     830
01H8DZZ71SYRKB4868B0WHV68E     510
01H8K9XWK972V31SN3HDRGY022     331
01H7S9MDFE2FW13DPP93DV0WWP     242
                              ... 
01JBTQQPYT7NYMAARHD9Y8PEAG       1
01JBTRNEQDW9RHAQR9F12S9VCJ       1
01JBTWH2Q6SNKQRKBFRFKCFKG6       1
01JBTWH7N4MYK2FS57SA3BFJAH       1
01JV6DBGH9Y0F51GFJ4KGKEC9K       1
Name: count, Length: 66504, dtype: int64

# Unique한 고객들과 동행한 횟수

In [23]:
from collections import defaultdict
from itertools import combinations

# 세션별로 customerId 리스트 만듦
session_groups = df.groupby('sessionId')['customerId'].apply(list)

# 고객별 동행인 저장 딕셔너리
co_diners = defaultdict(set)

# 세션별로 모든 고객 조합 생성
for customers in session_groups:
    for c1, c2 in combinations(set(customers), 2):
        co_diners[c1].add(c2)
        co_diners[c2].add(c1)

# 고객별 unique 동행인 수 계산
co_diner_counts = {customer: len(others) for customer, others in co_diners.items()}

# 상위 10명 출력
pd.Series(co_diner_counts).sort_values(ascending=False)


# 01JASCSJ4GAB7W730WT94BVA4C 라는 고객은 지금까지 총 2,604명의 서로 다른 고객들과 동행한 적이 있다는 뜻입니다.

01JASCSJ4GAB7W730WT94BVA4C    2604
01JDK73M1X4CMV8BFQYBS4ME2M    1504
01JDK73QWE570V6FAZSQVBTJJ8    1504
01JDK69D43TZR9RBT65V69WWG7    1504
01JDK73PNJ997KCKDN87B0D7XY    1504
                              ... 
01JD9MX4FKXTKY07JP3B3NRA2P       1
01JD9MX2CPWQEG6NWW8JXK6F55       1
01JD920G2PYAR33WKX24NBV02P       1
01JD91WFBKZ1FKAFPNKMSANA9B       1
01JEDSC9DQW49AZ8Z7YCZB3ZX8       1
Length: 30180, dtype: int64

# sessionid에 등장한 횟수

In [26]:
# 각 session에 여러 명이 있었는지 확인
multi_diner_sessions = df.groupby('sessionId').filter(lambda x: len(set(x['customerId'])) > 1)

# 각 customerId가 몇 번 동행 세션에 있었는지 count
co_dining_frequency = multi_diner_sessions['customerId'].value_counts()
# 동행 세션 참여 횟수가 2회 이상인 고객만 필터링
#co_dining_filtered = co_dining_frequency[co_dining_frequency > 1]

co_dining_frequency
# 다른 고객과 함께 있었던 세션에 총 644번 참여했음

customerId
01GYNFXQNDD0ATWHDFXCBHD5XG    644
01H7HNC3GKV65TAG2H3SGW7A0H    248
01H8DZZ71SYRKB4868B0WHV68E    131
01H847J2DGX5KEKMX386TKYGY1     70
01H7S9MDFE2FW13DPP93DV0WWP     53
                             ... 
01J9860CQAH8JQAT1WC9ZZQVHK      1
01HF937TWAFA3W29FRQ64K0ZXF      1
01J96FWJZWT17ZWNKGBVMP23W6      1
01J96FWF2WMY4FZSH7N6DS3F2W      1
01JV6CM29T5HDSGVQPFP8ZA0K0      1
Name: count, Length: 30180, dtype: int64

# 가장 많은 인원이 있는 세션은?

In [34]:
# sessionId별 고객 수 계산
session_sizes = df.groupby('sessionId')['customerId'].nunique()

# 상위 10개 확인 (가장 많은 인원이 있었던 세션)
session_sizes.sort_values(ascending=False)

sessionId
01JDK5ARYD1QYN83K263MTRSM9                                          1505
perf_test_initial_session                                           1097
01JDRS6PYM1M51PC343NYKA8M6                                           102
01HE7FVW7VKH21KPKWJM7P358S                                            36
01HESNEQNHBAPMYPEVBV9ZW95M                                            36
                                                                    ... 
01JB61B2YG3WHGFC4MJM1PKR2F                                             1
01JB61BK27J4NEJ38KJ2PH0HZX                                             1
01JB61CDXRMW33PV95NCDCRMTT                                             1
01JB61CJD3N04094Y1G6VVFE9E                                             1
wQIY0aU1NjcfJY0vQjGKbKYDTjsEH6dhEuBvdyOOSZAEx6cRDD9SC71grCHIFVgw       1
Name: customerId, Length: 74717, dtype: int64

# 누구와 몇 번 함께 하는가?

In [33]:
from itertools import combinations
from collections import Counter

# sessionId별 customerId 리스트
session_groups = df.groupby('sessionId')['customerId'].apply(list)

# 고객 쌍 누적용 카운터
pair_counter = Counter()

# 각 세션에서 2인 조합을 만들어 카운트
for customers in session_groups:
    customer_set = set(customers)
    for c1, c2 in combinations(sorted(customer_set), 2):
        pair_counter[(c1, c2)] += 1

# 결과를 DataFrame으로 변환
pair_df = pd.DataFrame([
    {'customer1': k[0], 'customer2': k[1], 'timesTogether': v}
    for k, v in pair_counter.items()
])

# 동행 횟수 기준 정렬 (선택)
pair_df = pair_df.sort_values(by='timesTogether', ascending=False)

# 상위 10쌍 확인
pair_df.head(10)


,customer1,customer2,timesTogether
40,01GYNFXQNDD0ATWHDFXCBHD5XG,01H7HNC3GKV65TAG2H3SGW7A0H,19
267,01H7HNC3GKV65TAG2H3SGW7A0H,01H8DZZ71SYRKB4868B0WHV68E,16
9684,01H7HNC3GKV65TAG2H3SGW7A0H,01H847J2DGX5KEKMX386TKYGY1,14
10092,01J1V0DRD1RMPF5JFPP2CKCC83,01J1VG1118TSY4KR1R6PJYK3V9,13
274,01H7S0P3HK62VATNYDHADQPHKJ,01H8DZZ71SYRKB4868B0WHV68E,12
5627,01H7S9MDFE2FW13DPP93DV0WWP,01HF3DR2VBV3D2M0SB6M32WXHQ,11
1893,01H7HNC3GKV65TAG2H3SGW7A0H,01H9MSJKX4CCDBG2JK327BSP84,11
428,01H7S0P3HK62VATNYDHADQPHKJ,01H98VZ8Y98MJ489TEJW7QCDS4,10
431,01H8DZZ71SYRKB4868B0WHV68E,01H98VZ8Y98MJ489TEJW7QCDS4,10
10326,01H8DZZ71SYRKB4868B0WHV68E,01J30PKXHBSFTMWEMHPZXZGZP3,10


In [35]:
# unique한 동행자 수를 통해 key diner 판별하기
from collections import defaultdict

co_diners = defaultdict(set)
for customers in session_groups:
    for c1, c2 in combinations(set(customers), 2):
        co_diners[c1].add(c2)
        co_diners[c2].add(c1)

unique_co_diner_counts = pd.Series({k: len(v) for k, v in co_diners.items()}, name='unique_co_diners')

pair_df['c1_co_diners'] = pair_df['customer1'].map(unique_co_diner_counts)
pair_df['c2_co_diners'] = pair_df['customer2'].map(unique_co_diner_counts)

def pick_key_diner(row):
    if row['c1_co_diners'] > row['c2_co_diners']:
        return row['customer1']
    elif row['c1_co_diners'] < row['c2_co_diners']:
        return row['customer2']
    else:
        return 'equal'  # 동점인 경우

pair_df['likely_key_diner'] = pair_df.apply(pick_key_diner, axis=1)

pair_df[['customer1', 'customer2', 'timesTogether', 'c1_co_diners', 'c2_co_diners', 'likely_key_diner']].head(10)



,customer1,customer2,timesTogether,c1_co_diners,c2_co_diners,likely_key_diner
40,01GYNFXQNDD0ATWHDFXCBHD5XG,01H7HNC3GKV65TAG2H3SGW7A0H,19,452,166,01GYNFXQNDD0ATWHDFXCBHD5XG
267,01H7HNC3GKV65TAG2H3SGW7A0H,01H8DZZ71SYRKB4868B0WHV68E,16,166,95,01H7HNC3GKV65TAG2H3SGW7A0H
9684,01H7HNC3GKV65TAG2H3SGW7A0H,01H847J2DGX5KEKMX386TKYGY1,14,166,74,01H7HNC3GKV65TAG2H3SGW7A0H
10092,01J1V0DRD1RMPF5JFPP2CKCC83,01J1VG1118TSY4KR1R6PJYK3V9,13,3,2,01J1V0DRD1RMPF5JFPP2CKCC83
274,01H7S0P3HK62VATNYDHADQPHKJ,01H8DZZ71SYRKB4868B0WHV68E,12,31,95,01H8DZZ71SYRKB4868B0WHV68E
5627,01H7S9MDFE2FW13DPP93DV0WWP,01HF3DR2VBV3D2M0SB6M32WXHQ,11,45,36,01H7S9MDFE2FW13DPP93DV0WWP
1893,01H7HNC3GKV65TAG2H3SGW7A0H,01H9MSJKX4CCDBG2JK327BSP84,11,166,33,01H7HNC3GKV65TAG2H3SGW7A0H
428,01H7S0P3HK62VATNYDHADQPHKJ,01H98VZ8Y98MJ489TEJW7QCDS4,10,31,20,01H7S0P3HK62VATNYDHADQPHKJ
431,01H8DZZ71SYRKB4868B0WHV68E,01H98VZ8Y98MJ489TEJW7QCDS4,10,95,20,01H8DZZ71SYRKB4868B0WHV68E
10326,01H8DZZ71SYRKB4868B0WHV68E,01J30PKXHBSFTMWEMHPZXZGZP3,10,95,1,01H8DZZ71SYRKB4868B0WHV68E


In [36]:
df.head()

,sessionId,customerId
0,01GZWETRHAHNKXQR2PDRP1ZEJC,01GYNFXQNDD0ATWHDFXCBHD5XG
1,01GZWETRHAHNKXQR2PDRP1ZEJC,01GYNFXQNDD0ATWHDFXCBHD5XG
2,01GZWG6V85M35RWMVVT0R8BW2T,01GYNFXQNDD0ATWHDFXCBHD5XG
3,01GZWKZ87MX12B4PFNZN2RDMPH,01GZWKZZJXDT4H4SAZ068CKYPR
4,01GZWKZ87MX12B4PFNZN2RDMPH,01GYNFXQNDD0ATWHDFXCBHD5XG


In [37]:
import pandas as pd

df = pd.DataFrame({
    'sessionId': [...],
    'customerId': [...]
})


In [38]:
df[['sessionId', 'customerId']].to_csv('orders_session_customer.csv', index=False)

In [39]:
import os
os.getcwd()


'/mnt/c/fintech/sicpama'

In [42]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host='127.0.0.1',
    database='orders',
    user='dev',
    password='pwd'
)

df = pd.read_sql("SELECT sessionId, customerId FROM orders", conn)
df.to_csv('orders_session_customer.csv', index=False)

/tmp/ipykernel_2377/4121208221.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT sessionId, customerId FROM orders", conn)


PermissionError: [Errno 13] Permission denied: 'orders_session_customer.csv'

In [43]:
import os
print(os.getcwd())


/mnt/c/fintech/sicpama


In [45]:
df[['sessionId', 'customerId']].to_csv('orders_session_customer.csv', index=False)